<a href="https://colab.research.google.com/github/2303A51780/Creditcarddeliquency/blob/main/Creditcarddeliquency_predection.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# CREDIT CARD DELINQUENCY PREDICTION - FULL PIPELINE
# (Clean, internship-ready code for Geldium case study)
# ============================================================

import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer

from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
    classification_report
)

# 1. LOAD DATA
# ------------------------------------------------------------

FILE_PATH = "/content/Delinquency_prediction_dataset (1).csv"


df = pd.read_csv(FILE_PATH)

print("Shape of dataset:", df.shape)
print("\nFirst 5 rows:")
print(df.head())

print("\nDataset info:")
print(df.info())

print("\nSummary statistics (numerical):")
print(df.describe())

print("\nMissing values per column:")
print(df.isnull().sum())

print("\nTarget distribution (Delinquent_Account):")
print(df["Delinquent_Account"].value_counts(normalize=True))

# ------------------------------------------------------------
# 2. FEATURE / TARGET SPLIT
# ------------------------------------------------------------

TARGET_COL = "Delinquent_Account"

# Drop Customer_ID because it's just an identifier, not a predictor
X = df.drop(columns=[TARGET_COL, "Customer_ID"])
y = df[TARGET_COL]

print("\nFeatures shape:", X.shape)
print("Target shape:", y.shape)

# 3. DEFINE NUMERICAL & CATEGORICAL COLUMNS
# ------------------------------------------------------------

numerical_cols = [
    "Age",
    "Income",
    "Credit_Score",
    "Credit_Utilization",
    "Missed_Payments",
    "Loan_Balance",
    "Debt_to_Income_Ratio",
    "Account_Tenure"
]

categorical_cols = [
    "Employment_Status",
    "Credit_Card_Type",
    "Location",
    "Month_1",
    "Month_2",
    "Month_3",
    "Month_4",
    "Month_5",
    "Month_6"
]

# Safety: keep only columns that actually exist in X
numerical_cols = [c for c in numerical_cols if c in X.columns]
categorical_cols = [c for c in categorical_cols if c in X.columns]

print("\nNumerical columns used:", numerical_cols)
print("Categorical columns used:", categorical_cols)

# 4. PREPROCESSING PIPELINE
# - Impute missing values
# - Scale numerical features
# - One-hot encode categorical features
# ------------------------------------------------------------

numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numerical_cols),
        ("cat", categorical_transformer, categorical_cols)
    ],
    remainder="drop"
)

# 5. TRAIN / TEST SPLIT
# ------------------------------------------------------------

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("\nTrain shape:", X_train.shape, "Test shape:", X_test.shape)
print("Train target distribution:")
print(y_train.value_counts(normalize=True))

# 6. EVALUATION HELPER
# ------------------------------------------------------------

def evaluate_model(name, y_true, y_pred, y_proba=None):
    print(f"\n================= {name} Evaluation =================")
    print("Accuracy :", round(accuracy_score(y_true, y_pred), 4))
    print("Precision:", round(precision_score(y_true, y_pred, zero_division=0), 4))
    print("Recall   :", round(recall_score(y_true, y_pred, zero_division=0), 4))
    print("F1 Score :", round(f1_score(y_true, y_pred, zero_division=0), 4))

    if y_proba is not None:
        try:
            auc = roc_auc_score(y_true, y_proba)
            print("AUC-ROC  :", round(auc, 4))
        except Exception as e:
            print("AUC-ROC  : could not be calculated -", e)

    print("\nConfusion Matrix:")
    print(confusion_matrix(y_true, y_pred))

    print("\nClassification Report:")
    print(classification_report(y_true, y_pred, zero_division=0))


# 7. MODEL 1 – LOGISTIC REGRESSION (WITH CLASS WEIGHT)
# ------------------------------------------------------------

log_reg_model = Pipeline(steps=[
    ("preprocess", preprocessor),
    ("clf", LogisticRegression(
        max_iter=1000,
        class_weight="balanced",   # handle class imbalance
        solver="lbfgs"
    ))
])

print("\nTraining Logistic Regression model...")
log_reg_model.fit(X_train, y_train)

y_pred_lr = log_reg_model.predict(X_test)
y_proba_lr = log_reg_model.predict_proba(X_test)[:, 1]

evaluate_model("Logistic Regression", y_test, y_pred_lr, y_proba_lr)

# 8. MODEL 2 – DECISION TREE (INTERPRETABLE)
# ------------------------------------------------------------

tree_model = Pipeline(steps=[
    ("preprocess", preprocessor),
    ("clf", DecisionTreeClassifier(
        max_depth=5,
        class_weight="balanced",
        random_state=42
    ))
])

print("\nTraining Decision Tree model...")
tree_model.fit(X_train, y_train)

y_pred_tree = tree_model.predict(X_test)
y_proba_tree = tree_model.predict_proba(X_test)[:, 1]

evaluate_model("Decision Tree", y_test, y_pred_tree, y_proba_tree)

# 9. WHICH FEATURES DRIVE DELINQUENCY?
#    (Feature importance + coefficients)
# ------------------------------------------------------------

# Get feature names from the preprocessor
preprocess_fitted = tree_model.named_steps["preprocess"]

# Numeric feature names (as-is)
num_features = numerical_cols

# One-hot encoded categorical feature names
ohe = preprocess_fitted.named_transformers_["cat"].named_steps["onehot"]
ohe_features = list(ohe.get_feature_names_out(categorical_cols))

all_feature_names = num_features + ohe_features

# ---- Decision Tree Feature Importances ----
dt_importances = tree_model.named_steps["clf"].feature_importances_

importance_df = pd.DataFrame({
    "Feature": all_feature_names,
    "Importance": dt_importances
}).sort_values(by="Importance", ascending=False)

print("\n=========== Top 10 Predictors of Delinquency (Decision Tree) ===========")
print(importance_df.head(10))

# ---- Logistic Regression Coefficients ----
lr_coef = log_reg_model.named_steps["clf"].coef_[0]

coef_df = pd.DataFrame({
    "Feature": all_feature_names,
    "Coefficient": lr_coef
}).sort_values(by="Coefficient", ascending=False)

print("\n=========== Features that INCREASE Delinquency Risk (Top 10) ===========")
print(coef_df.head(10))

print("\n=========== Features that DECREASE Delinquency Risk (Top 10) ===========")
print(coef_df.tail(10))

# 10. EXAMPLE: RISK SCORING FOR ONE CUSTOMER
# ------------------------------------------------------------

example_row = X_test.iloc[[0]]   # keep as DataFrame

print("\nExample customer from test set:")
print(example_row)

example_pred = log_reg_model.predict(example_row)[0]
example_proba = log_reg_model.predict_proba(example_row)[0, 1]

print("\nPredicted delinquency class (0 = No, 1 = Yes):", example_pred)
print("Predicted delinquency probability:", round(example_proba, 3))

if example_proba >= 0.7:
    risk_band = "High Risk"
elif example_proba >= 0.4:
    risk_band = "Medium Risk"
else:
    risk_band = "Low Risk"

print("Assigned Risk Band:", risk_band)


Shape of dataset: (500, 19)

First 5 rows:
  Customer_ID  Age    Income  Credit_Score  Credit_Utilization  \
0    CUST0001   56  165580.0         398.0            0.390502   
1    CUST0002   69  100999.0         493.0            0.312444   
2    CUST0003   46  188416.0         500.0            0.359930   
3    CUST0004   32  101672.0         413.0            0.371400   
4    CUST0005   60   38524.0         487.0            0.234716   

   Missed_Payments  Delinquent_Account  Loan_Balance  Debt_to_Income_Ratio  \
0                3                   0       16310.0              0.317396   
1                6                   1       17401.0              0.196093   
2                0                   0       13761.0              0.301655   
3                3                   0       88778.0              0.264794   
4                2                   0       13316.0              0.510583   

  Employment_Status  Account_Tenure Credit_Card_Type     Location Month_1  \
0             